In [8]:
# Setup: connect to v.db (read-only), load the FULL set of "scope" errors
# (2,668,388 rows) -- this is the complete population, not a sample.
import sqlite3
import pandas as pd
import re

DB_PATH = "/Users/anil/code/reroll-data/data/v.db"
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
sample_df = pd.read_sql_query(
    """
    SELECT project, filename, reroll_error
    FROM repodata_conversion
    WHERE reroll_error LIKE 'scope:%'
    """,
    con,
)
len(sample_df), sample_df.head()

(2668388,
           project                                 filename  \
 0   0-core-client   0_core_client-1.1.0a3-py3-none-any.whl   
 1   0-core-client   0_core_client-1.1.0a5-py3-none-any.whl   
 2   0-core-client   0_core_client-1.1.0a7-py3-none-any.whl   
 3   0-core-client   0_core_client-1.1.0a8-py3-none-any.whl   
 4  0-orchestrator  0_orchestrator-1.1.0a0-py3-none-any.whl   
 
                                         reroll_error  
 0  scope: UnsupportedPrereleaseError: rejected pr...  
 1  scope: UnsupportedPrereleaseError: rejected pr...  
 2  scope: UnsupportedPrereleaseError: rejected pr...  
 3  scope: UnsupportedPrereleaseError: rejected pr...  
 4  scope: UnsupportedPrereleaseError: rejected pr...  )

In [9]:
# Alias to a clearer name -- this is the full population of scope errors,
# not a sample (cheap reference assignment, no re-query).
scope_df = sample_df
len(scope_df)

2668388

In [10]:
# What exception types make up "scope" errors (full population)?
exc_type = scope_df["reroll_error"].str.extract(r"^scope:\s*(\w+)")
scope_df["exc_type"] = exc_type[0]
scope_df["exc_type"].value_counts()

exc_type
UnsupportedPrereleaseError            1463549
UnsupportedPlatformError              1053322
UnsupportedInterpreterError            146575
UnsupportedInterpreterVersionError       4942
Name: count, dtype: int64

In [11]:
# Flag filenames that look like pypy or python-2.x wheels, based on the wheel
# filename tag convention: {name}-{version}-{python tag}-{abi tag}-{platform}.whl
# python tag examples: pp27/pp36 (pypy), cp27 (cpython 2.7), py2, py27
pypy_or_py2 = scope_df["filename"].str.contains(r"-(?:pp\d+|cp2\d|py2\d?)-", regex=True)
scope_df["pypy_or_py2"] = pypy_or_py2

summary = scope_df.groupby("exc_type")["pypy_or_py2"].agg(["sum", "count", "mean"])
summary["mean"] = (summary["mean"] * 100).round(1)
summary.columns = ["pypy_or_py2_count", "n", "pct_pypy_or_py2"]
summary["pct_of_all_scope"] = (summary["n"] / len(scope_df) * 100).round(1)
print(
    f"Overall: {pypy_or_py2.sum():,} / {len(scope_df):,} "
    f"({pypy_or_py2.mean() * 100:.1f}%) scope-error filenames look like pypy/py2"
)
summary.sort_values("n", ascending=False)

Overall: 196,847 / 2,668,388 (7.4%) scope-error filenames look like pypy/py2


,pypy_or_py2_count,n,pct_pypy_or_py2,pct_of_all_scope
exc_type,,,,
UnsupportedPrereleaseError,50363,1463549,3.4,54.8
UnsupportedPlatformError,0,1053322,0.0,39.5
UnsupportedInterpreterError,146484,146575,99.9,5.5
UnsupportedInterpreterVersionError,0,4942,0.0,0.2


In [12]:
# Spot-check the prerelease rows that also matched the pypy/py2 regex --
# are these real py2/pypy wheels, or false positives in the regex?
print(
    f"{summary.loc['UnsupportedPrereleaseError', 'pypy_or_py2_count']:,.0f} of the "
    f"{summary.loc['UnsupportedPrereleaseError', 'n']:,.0f} prerelease rejections "
    "also look like pypy/py2 wheels -- these get masked by the prerelease check "
    "before the interpreter check ever runs."
)
scope_df[
    (scope_df["exc_type"] == "UnsupportedPrereleaseError") & scope_df["pypy_or_py2"]
]["filename"].head(10).tolist()

50,363 of the 1,463,549 prerelease rejections also look like pypy/py2 wheels -- these get masked by the prerelease check before
the interpreter check ever runs.


['3gpp_citations-0.1.dev0-py2-none-any.whl',
 'AGLOW-0.1.0rc1-py2-none-any.whl',
 'AGLOW-0.1.0rc2-py2-none-any.whl',
 'AGLOW-0.1.0rc3-py2-none-any.whl',
 'AGLOW-0.1.0rc4-py2-none-any.whl',
 'ARCCSSive-0.2.2.dev34-py2-none-any.whl',
 'ARS-0.5a2-py27-none-any.whl',
 'ARS-0.5b1-py27-none-any.whl',
 'AccessControl-4.0b2-cp27-cp27m-win32.whl',
 'AccessControl-4.0b2-cp27-cp27m-win_amd64.whl']

In [13]:
# Dig into UnsupportedPlatformError (39.5% of scope errors, 0% pypy/py2 --
# --allow-pre can't touch this bucket at all). Extract the platform tag from
# the wheel filename (second-to-last dash-separated field, minus .whl) to see
# what's actually driving these rejections.
platform_errs = scope_df[scope_df["exc_type"] == "UnsupportedPlatformError"].copy()


def extract_platform_tag(filename: str) -> str:
    # {name}-{version}[-{build}]-{python tag}-{abi tag}-{platform tag}.whl
    stem = filename[: -len(".whl")]
    return stem.rsplit("-", 1)[-1]


platform_errs["platform_tag"] = platform_errs["filename"].apply(extract_platform_tag)
print(f"{len(platform_errs):,} UnsupportedPlatformError rows")
platform_errs["platform_tag"].value_counts().head(30)

1,053,322 UnsupportedPlatformError rows


platform_tag
win32                                                                        207546
musllinux_1_2_x86_64                                                         138273
musllinux_1_2_aarch64                                                         94606
musllinux_1_2_i686                                                            59053
musllinux_1_1_x86_64                                                          57885
manylinux_2_17_ppc64le.manylinux2014_ppc64le                                  55121
manylinux_2_17_armv7l.manylinux2014_armv7l                                    52918
manylinux_2_17_i686.manylinux2014_i686                                        51226
manylinux_2_17_s390x.manylinux2014_s390x                                      50481
musllinux_1_2_armv7l                                                          36850
musllinux_1_1_i686                                                            30105
manylinux_2_5_i686.manylinux1_i686                             

In [14]:
# Bucket every platform tag into a human reason, using reroll's documented
# exclusions (docs/wheel_filename.md): musllinux, 32-bit OSes, iOS/Android,
# non-manylinux-glibc linux, non-x86_64/arm64/universal2 mac, and unsupported
# (non-x86_64/aarch64) architectures on otherwise-valid manylinux tags.


def bucket_platform(tag: str) -> str:
    if "musllinux" in tag:
        return "musllinux (musl libc)"
    if tag == "win32" or tag == "win_ia64":
        return "32-bit / non-amd64-arm64 windows"
    if "ios_" in tag or "iphoneos" in tag:
        return "iOS"
    if "android" in tag:
        return "Android"
    if "emscripten" in tag or "wasi" in tag or "wasm" in tag:
        return "wasm (emscripten/wasi)"
    if tag.startswith("macosx"):
        if re.search(r"(x86_64|arm64|universal2)$", tag):
            return "macOS (supported arch, other reason)"
        return "macOS unsupported arch (intel/ppc/fat/i386)"
    if tag.startswith("win"):
        return "windows other"
    if tag.startswith("manylinux") or tag.startswith("linux"):
        if re.search(r"(i686|i386)", tag):
            return "linux 32-bit (i686)"
        if re.search(r"(ppc64le|ppc64|s390x|armv7l|riscv64|loongarch64|mips)", tag):
            return "linux unsupported arch (ppc64le/s390x/armv7l/etc.)"
        if tag.startswith("linux_"):
            return "bare linux_* (non-manylinux glibc tag)"
        return "linux other"
    return "other/unrecognized"


platform_errs["reason"] = platform_errs["platform_tag"].apply(bucket_platform)
reason_counts = platform_errs["reason"].value_counts()
reason_pct = (reason_counts / len(platform_errs) * 100).round(1)
pd.DataFrame({"count": reason_counts, "pct_of_platform_errors": reason_pct})

,count,pct_of_platform_errors
reason,,
musllinux (musl libc),463205,44.0
32-bit / non-amd64-arm64 windows,207577,19.7
linux unsupported arch (ppc64le/s390x/armv7l/etc.),194145,18.4
linux 32-bit (i686),173157,16.4
macOS unsupported arch (intel/ppc/fat/i386),11719,1.1
bare linux_* (non-manylinux glibc tag),1394,0.1
Android,1277,0.1
iOS,456,0.0
wasm (emscripten/wasi),384,0.0


In [15]:
pd.set_option("display.max_colwidth", 60)
summary_platform = pd.DataFrame(
    {"count": reason_counts, "pct_of_platform_errors": reason_pct}
)
for reason, row in summary_platform.iterrows():
    print(f"{row['count']:>9,.0f}  {row['pct_of_platform_errors']:>5.1f}%  {reason}")

  463,205   44.0%  musllinux (musl libc)
  207,577   19.7%  32-bit / non-amd64-arm64 windows
  194,145   18.4%  linux unsupported arch (ppc64le/s390x/armv7l/etc.)
  173,157   16.4%  linux 32-bit (i686)
   11,719    1.1%  macOS unsupported arch (intel/ppc/fat/i386)
    1,394    0.1%  bare linux_* (non-manylinux glibc tag)
    1,277    0.1%  Android
      456    0.0%  iOS
      384    0.0%  wasm (emscripten/wasi)
        8    0.0%  windows other


### Deep dive: `UnsupportedPlatformError` (39.5% of scope errors, 1,053,322 rows)

You're right that `--allow-pre` does nothing for this bucket -- it's a
completely separate rejection path, and **0% of it is pypy/py2** because it's
purely about platform/arch tags, independent of the interpreter tag.

Traced `reroll`'s `classify_platform()` (`src/reroll/filename/platform.py:20-52`,
raised from `src/reroll/filename/wheel_config.py:88`). Unlike `allow_pre`,
**this is not a runtime toggle at all** -- it's a hard-coded regex allowlist
with no config knob:
- Only `x86_64` and `arm64`/`aarch64` architectures are supported (`src/reroll/filename/enums.py`'s `Arch` enum has exactly two members).
- Only `manylinux_*`/`manylinux{1,2010,2014}` (glibc), `macosx_*_{x86_64,arm64,universal2}`, and `win_{amd64,arm64}` platform families are recognized.
- This scope is explicitly documented in `docs/wheel_filename.md:8-14`: reroll
  will not support musllinux, 32-bit OSes, iOS/Android, non-manylinux-glibc
  linux, or non-x86_64/arm64/universal2 mac builds -- "support can be added
  later based on real use cases." It's been this way since the very first
  filename-parsing implementation (commit `26d7440`); never loosened or
  tightened since.

**Breakdown of the 1.05M platform rejections in our data:**

| reason | count | % of platform errors |
|---|---|---|
| musllinux (musl libc) | 463,205 | 44.0% |
| 32-bit / non-amd64-arm64 windows (`win32`) | 207,577 | 19.7% |
| linux unsupported arch (ppc64le/s390x/armv7l/...) | 194,145 | 18.4% |
| linux 32-bit (i686) | 173,157 | 16.4% |
| macOS unsupported arch (intel/ppc/fat/i386) | 11,719 | 1.1% |
| bare `linux_*` (non-manylinux glibc tag) | 1,394 | 0.1% |
| Android | 1,277 | 0.1% |
| iOS | 456 | 0.0% |
| wasm (emscripten/wasi) | 384 | 0.0% |

Every one of these is a legitimate, currently-out-of-scope platform per
reroll's own docs -- there is no bug in reroll here, and there's no flag
`reroll-data` is failing to pass through, because none exists.

### Recommendation

Unlike the prerelease bucket, this isn't fixable by wiring up an existing
flag -- widening platform/arch support requires an actual reroll library
change (extending `classify_platform`'s regexes, `Arch` enum, and the
subdir-mapping table in `subdir.py`). Given the volume, in priority order:

1. **musllinux (44%, 463k wheels)** is by far the largest chunk. musl-based
   Linux is well-supported by conda-forge today (Alpine/musl builds exist),
   so this looks like the highest-leverage candidate for a future reroll
   scope expansion -- worth filing as a feature request against reroll
   itself (per its own docs: *"musl platform tags... may be revisited if
   conda ever supports these variants"* -- conda-forge already does, this
   may be stale reasoning).
2. **`win32` (19.7%, 208k wheels)** and **i686 (16.4%, 173k)** are 32-bit
   legacy architectures reroll deliberately excludes; conda itself has been
   dropping 32-bit platform support broadly, so these are likely permanently
   out of scope and not worth chasing.
3. **ppc64le/s390x/armv7l (18.4%, 194k)** are minority-architecture Linux
   builds; low priority unless there's a specific downstream consumer need.
4. The remaining categories (mac unsupported arch, iOS, Android, wasm) are
   under 2% combined and not worth prioritizing.

Net: after the `allow_pre` fix resolves the prerelease bucket, the platform
bucket will still legitimately account for ~1.05M "scope" rejections -- that
volume is real and expected, not a bug, unless/until reroll's platform
allowlist is deliberately expanded (musllinux being the strongest
candidate).

In [16]:
# How many distinct packages actually account for the 463,205 musllinux rows?
musl = platform_errs[platform_errs["reason"] == "musllinux (musl libc)"]
n_projects = musl["project"].nunique()
print(f"{len(musl):,} musllinux wheel rows from {n_projects:,} distinct projects")
print(f"-> average {len(musl) / n_projects:.1f} musllinux wheels per project")

top_projects = musl["project"].value_counts()
top_projects.head(20)

463,205 musllinux wheel rows from 6,000 distinct projects
-> average 77.2 musllinux wheels per project


project
ddtrace                 8308
aioesphomeapi           4038
stringzilla             3417
dbus-fast               2663
simsimd                 2631
aiohttp                 2041
clickhouse-connect      1982
zeroconf                1981
pydantic_core           1923
habluetooth             1906
coverage                1802
bitarray                1744
RapidFuzz               1741
ruff                    1644
attoworld               1608
yarl                    1395
regex                   1376
passagemath-coxeter3    1178
granian                 1157
psycopg-binary          1156
Name: count, dtype: int64

In [17]:
# Why 77 musllinux wheels/project on average? Check the shape of the
# distribution, and how much of it is just multiple releases x multiple
# python versions x multiple musl tags (1_1 vs 1_2) x multiple arches for
# the same handful of popular C-extension packages.
print("Distribution of musllinux-row-count per project:")
print(top_projects.describe())
print()
print("Cumulative share of rows held by the top N projects:")
cum_share = top_projects.cumsum() / len(musl) * 100
for n in [10, 50, 100, 500, 1000, 6000]:
    print(
        f"  top {n:>5,} projects -> {cum_share.iloc[min(n, len(cum_share)) - 1]:.1f}% of musllinux rows"
    )

# Sanity check the "many wheels per release" theory for the #1 project
ddtrace = musl[musl["project"] == "ddtrace"]
print(
    f"\nddtrace: {len(ddtrace):,} musllinux rows, "
    f"{ddtrace['filename'].str.extract(r'-(\d+\.\d+\.\d+)')[0].nunique()} distinct versions seen in filenames"
)

Distribution of musllinux-row-count per project:
count    6000.000000
mean       77.200833
std       198.152830
min         1.000000
25%        10.000000
50%        28.000000
75%        74.000000
max      8308.000000
Name: count, dtype: float64

Cumulative share of rows held by the top N projects:
  top    10 projects -> 6.7% of musllinux rows
  top    50 projects -> 16.0% of musllinux rows
  top   100 projects -> 23.4% of musllinux rows
  top   500 projects -> 51.9% of musllinux rows
  top 1,000 projects -> 68.1% of musllinux rows
  top 6,000 projects -> 100.0% of musllinux rows

ddtrace: 8,308 musllinux rows, 492 distinct versions seen in filenames


### musllinux: 463,205 rows come from only 6,000 distinct projects

- Median: 28 musllinux wheel rows/project; mean 77.2 (heavily right-skewed,
  max 8,308).
- This isn't 6,000 packages each shipping one wheel -- it's a moderate set
  of C-extension packages (musl wheels only exist for compiled/native code)
  each shipping musllinux wheels across **many releases x multiple CPython
  versions x multiple musl libc versions (1_1/1_2) x multiple arches**.
  `ddtrace` alone accounts for 8,308 rows across 492 distinct versions --
  it's an actively-released package that has consistently published
  musllinux wheels for years.
- Concentration: top 10 projects = 6.7% of rows, top 500 (~8% of projects)
  = 51.9%, top 1,000 (~17% of projects) = 68.1%. Not dominated by a handful
  of outliers, but not flat either -- a long tail of packages each
  contributing dozens of wheels over their release history.
- Top projects are exactly what you'd expect to see: `ddtrace`,
  `aioesphomeapi`, `stringzilla`, `dbus-fast`, `simsimd`, `aiohttp`,
  `clickhouse-connect`, `zeroconf`, `pydantic_core`, `ruff`, `bitarray`,
  `regex` -- popular, frequently-released, natively-compiled libraries that
  have adopted musllinux (Alpine/musl support) as standard practice in
  their CI wheel-building matrix.

This confirms the 463k figure is real and not an artifact of double-counting
or a filename-parsing bug -- it's the natural result of ~6,000 actively
maintained native-extension packages each publishing musl wheels across
their full release history.

### Finding (full population, 2,668,388 scope-error rows -- not a sample)

| `scope` exception type              | count     | % of all scope errors | % w/ pypy/py2-looking filename |
|---|---|---|---|
| `UnsupportedPrereleaseError`         | 1,463,549 | 54.8% | 3.4%  |
| `UnsupportedPlatformError`           | 1,053,322 | 39.5% | 0.0%  |
| `UnsupportedInterpreterError`        |   146,575 |  5.5% | 99.9% |
| `UnsupportedInterpreterVersionError` |     4,942 |  0.2% | 0.0%  |

**196,847 / 2,668,388 (7.4%) of all scope-error filenames actually contain a
pypy/py2 tag** (`pp\d+`, `cp2\d`, or `py2\d?`), even after counting the
50,363 pypy/py2 wheels that got masked inside `UnsupportedPrereleaseError`
(rejected for being a pre-release *before* the interpreter check ever runs).

**Conclusion:** confirmed at full scale, not just in the sample.
`UnsupportedInterpreterError` (true pypy/non-cpython) is only 5.5% of scope
errors, not "mostly." The dominant scope-error cause is
`UnsupportedPrereleaseError` (54.8%, pre-release versions rejected because
`allow_pre` isn't set), followed by `UnsupportedPlatformError` (39.5%). Of
the 2.6M scope errors, only **~197k** are actual pypy/python2 wheels -- the
rest are being excluded for unrelated reasons (prerelease, platform) that
happen to fall under the same "scope" bucket.

## Root cause: `allow_pre` pass-through bug in reroll-data (not a reroll bug)

Traced the code path for the dominant `UnsupportedPrereleaseError` bucket
(54.8% of all 2.6M scope errors, confirmed on the full population) across
both repos:

**`~/code/reroll` (library) -- behaving as designed:**
- `UnsupportedPrereleaseError` (`src/reroll/errors.py:87-90`) is raised in
  `parse_filename()` (`src/reroll/filename/__init__.py:70-74`), *unconditionally
  before* the per-tag interpreter/platform loop that would otherwise raise
  `UnsupportedInterpreterError`/`UnsupportedPlatformError` (loop starts at
  line 84). This is why a wheel that is both prerelease *and* pypy/py2 always
  gets bucketed as "prerelease," masking the true interpreter-scope count.
- `allow_pre` defaults to `False` everywhere in the library on purpose --
  it's a plain keyword arg (reroll has no CLI of its own), and
  `docs/matchspec.md`'s "Pre-release" section explicitly frames it as a
  **caller-facing toggle**: *"an `allow_pre` flag can toggle this behavior"*
  for a repodata author who wants their channel to include alpha/beta/rc
  packages -- exactly reroll-data's bulk/archival use case.
- Conclusion: reroll's default-off behavior is intentional, documented, and
  not a bug.

**`~/code/reroll-data` (this repo, orchestration) -- the actual bug:**
- The only pipeline path that actually runs (`Makefile:190-191` ->
  `reroll-data repodata reroll-convert` -> `src/reroll_data/cli.py`) never
  defines an `--allow-pre` flag on the `reroll-convert` subparser, and
  `cmd_repodata_reroll_convert()` (`cli.py:246-253`) calls
  `_reroll_convert.convert(...)` **without** an `allow_pre` argument at all
  -- confirmed zero matches for `allow_pre`/`allow-pre` in `cli.py`.
- The plumbing to fix this already exists and is correct:
  `reroll_convert.convert()` (`reroll_convert.py:232-241`) already accepts
  `allow_pre: bool = False` and threads it correctly down to
  `reroll_index_demo._entry_from_db(..., allow_pre=_ALLOW_PRE)`.
  `reroll_convert.py` even has its own unused `main()` (lines 432-489) with a
  working `--allow-pre` CLI flag -- but that entry point isn't registered in
  `pyproject.toml` (`[project.scripts]` only exposes `cli:main` and
  `investigate:main`) and nothing in the Makefile/README invokes it. It's
  dead code.

### Recommendation
Wire up the flag that already exists one layer down, rather than building
anything new:
1. In `src/reroll_data/cli.py`, add `--allow-pre` (`action="store_true"`,
   default `False`) to the `reroll-convert` subparser, mirroring the
   already-correct flag definition in `reroll_convert.py:460-464`.
2. Pass `allow_pre=args.allow_pre` into the `_reroll_convert.convert(...)`
   call in `cmd_repodata_reroll_convert` (`cli.py:246-253`), mirroring
   `reroll_convert.py:484`.
3. Since this is a bulk/archival PyPI mirror (the exact use case
   `docs/matchspec.md` describes for turning `allow_pre` on), default the
   *pipeline's* invocation to `--allow-pre` on, or make it on by default in
   the Makefile target, so future full runs don't silently drop half of
   scope errors into the wrong bucket.
4. After the fix, re-run `reroll-convert --retry-errors` so the exact
   1,463,549 previously-misclassified prerelease rejections get reprocessed
   -- most should convert successfully, and the remainder will fall through
   to the correct `UnsupportedInterpreterError`/`UnsupportedPlatformError`
   buckets, giving an accurate scope-error picture (true pypy/py2 count would
   land around 197k, not 2.6M).

This is a one-line-of-plumbing fix, not a design change to reroll.

**Caveat:** this only fixes the `UnsupportedPrereleaseError` bucket (54.8%
of scope errors). It does **not** touch `UnsupportedPlatformError` (39.5%,
1.05M rows) -- see the platform deep-dive below, which is a separate, much
harder problem (no config flag exists; it requires an actual reroll library
change to widen the platform/arch allowlist, with musllinux being the
largest and most plausible candidate).